## Prompt Testing — Final Variable Identification

Two approaches to identify which dataset variables are actually used in analysis:
- **Prompt 1 (with AST)**: notebook code + AST-detected source variables
- **Prompt 2 (code only)**: notebook code only, LLM finds everything

Both should return the same output: the final filtered variable(s) for each dataset source.

## Setup

In [ ]:
import sys
import os
import json
import nbformat

sys.path.insert(0, '../../src')

from notebook_parser import parse_notebook, strip_ipython_directives
from llm import llm
from langchain_core.prompts import ChatPromptTemplate

## Helper — extract code from notebook

In [ ]:
def extract_code(notebook_path: str) -> str:
    """Extracts all code cells from a notebook as a single string."""
    with open(notebook_path) as f:
        nb = nbformat.read(f, as_version=4)
    cells = []
    for cell in nb.cells:
        if cell.cell_type == 'code':
            cleaned = strip_ipython_directives(cell.source)
            if cleaned.strip():
                cells.append(cleaned)
    return '\n\n'.join(cells)

## Prompt Definitions

In [ ]:
PROMPT_WITH_AST = ChatPromptTemplate.from_template(
"""You are analyzing a Jupyter notebook to identify which dataset variables are actually used in the scientific analysis.

I have already detected these dataset source variables using static analysis:

{ast_variables}

For each source variable, trace the data flow through the notebook and identify the final variable that holds the filtered or subsetted data actually used in the analysis. Scientists often load many datasets then filter by time range, resolution, variable type, or geography.

Rules:
- If no filtering occurs and the source variable is used directly, return the source variable itself.
- If the data flows through filtering operations (boolean indexing, subsetting, deduplication), return the last filtered variable before the data is consumed by analysis (PCA, plotting, modeling, etc.).
- If datasets are combined (e.g. ds3 = ds1 + ds2), only return the combined variable.
- Only return variables that hold dataset collections (DataFrames, dataset objects), not scalars, plots, or single rows.

Return ONLY a JSON object mapping each source variable to its final variable. No explanation.

Example: {{"df_res": "filtered_df2", "ds": "ds", "D_dir": "df_temp"}}

Notebook code:
```python
{code}
```""")

In [ ]:
PROMPT_CODE_ONLY = ChatPromptTemplate.from_template(
"""You are an expert at reading scientific Python notebooks and identifying where external datasets are loaded.

Analyze a Jupyter notebook to identify all dataset source variables that would yield citations.

A dataset source variable is a variable that directly loads or queries external data through one of these tools:
- PyLiPD: LiPD() objects that load data via load(), load_from_dir(), or load_remote_datasets()
- PyleoTUPS: PangaeaDataset() or NOAADataset() objects that search for studies via search_studies()
- LiPDGraph: SPARQL queries sent via requests.post() to the LinkedEarth endpoint (linkedearth.graphdb.mint.isi.edu), with results parsed into a DataFrame via pd.read_csv()
- Other libraries may also load external data — for example, xr.open_dataset(), pd.read_csv(url), cfr.ProxyDatabase.fetch(), or requests.get() downloading files — return these as well, paired with the library name used to load the data.

Rules:
- Return each dataset source variable paired with the tool it uses.
- If multiple LiPD objects are created (e.g. D1 = LiPD(), D2 = LiPD()), return each one separately — each may load different datasets with different citations.
- If datasets are combined (e.g. ds_sum = ds1 + ds2), return the individual source variables (ds1, ds2), not the combined variable.
- Ignore libraries used only for analysis or visualization (e.g. pyleoclim, matplotlib, numpy, scipy).
- Do not return derived variables like filtered DataFrames, plot objects, or scalars.

Return ONLY a JSON list of [variable, tool] pairs. No explanation.
Example: [["D", "PyLiPD"], ["ds", "PyleoTUPS"], ["df_res", "LiPDGraph"], ["iso_ds", "xarray"], ["intcal20", "pandas"]]

Notebook code:
```python
{code}
```""")

## Test Runner

In [ ]:
def run_test(notebook_path: str, expected: dict):
    """Runs both prompts on a notebook and compares to expected output."""
    code = extract_code(notebook_path)
    result = parse_notebook(notebook_path)

    # Format AST-detected variables for Prompt 1
    ast_lines = []
    for ds in result['datasets']:
        line = f"- {ds['variable']} ({ds['source_type']})"
        if 'class' in ds:
            line += f" — {ds['class']}"
        ast_lines.append(line)
    ast_str = '\n'.join(ast_lines) if ast_lines else '(none detected)'

    print(f'=== {os.path.basename(notebook_path)} ===')
    print(f'AST-detected: {[ds["variable"] for ds in result["datasets"]]}')
    print(f'Expected:     {expected}')
    print()

    # Prompt 1: with AST hints
    chain1 = PROMPT_WITH_AST | llm
    resp1 = chain1.invoke({'ast_variables': ast_str, 'code': code})
    print(f'Prompt 1 (with AST):  {resp1.content}')

    # Prompt 2: code only
    chain2 = PROMPT_CODE_ONLY | llm
    resp2 = chain2.invoke({'code': code})
    print(f'Prompt 2 (code only): {resp2.content}')
    print()

## Test Cases

| Notebook | Tests | Expected |
|---|---|---|
| PyleoTUPS | one library, adding DataFrames | ds, ds_sum, noaa_ds, pangaea_ds map to themselves |
| LIPD | one library, filtering | D_dir maps to df_temp or df_filt |
| paleoPCA | LiPDGraph, heavy filtering | df_res maps to filtered_df2 |
| dataset_pipeline | multiple extraction tools | D, ds, ds_noaa map to themselves |

In [ ]:
run_test('PyleoTUPS.ipynb', {
    'ds': 'ds',
    'ds_sum': 'ds_sum',
    'noaa_ds': 'noaa_ds',
    'pangaea_ds': 'pangaea_ds',
})

In [ ]:
run_test('LIPD.ipynb', {
    'D_remote': 'D_remote',
    'D_url': 'D_url',
    'D': 'D',
    'D_dir': 'df_temp',
    'D_src': 'D_reloaded',
    'D_reloaded': 'D_reloaded',
})

In [ ]:
run_test('paleoPCA.ipynb', {
    'df_res': 'filtered_df2',
})

In [ ]:
run_test('dataset_pipeline.ipynb', {
    'D': 'D',
    'ds': 'ds',
    'ds_noaa': 'ds_noaa',
})

---

## Exemplar-Based Prompts (In-Context Learning)

Instead of relying only on abstract rules, these prompts include a **worked example** — a real code snippet with its known-correct answer — so the LLM can learn the mapping pattern by example.

Three exemplars vary in **similarity** to the test notebook:

| Exemplar | Source | Similarity to LIPD.ipynb |
|---|---|---|
| PyLiPD | `LiPD()` load + DataFrame filtering | **High** — same library and patterns |
| PyleoTUPS | `PangaeaDataset()` / `NOAADataset()` search + combine | **Medium** — different library, simpler flow |
| LiPDGraph | SPARQL `requests.post()` + heavy DataFrame filtering | **Low** — no LiPD library, SQL-style query origin |

### Exemplar Snippets

Each snippet is a condensed version of a real test notebook, short enough to fit in a prompt but showing the key data-flow patterns (load → transform → filter → consume).

In [ ]:
EXEMPLAR_PYLIPD_CODE = """\
from pylipd.lipd import LiPD

D = LiPD()
D.load('Ocn-Palmyra.Nurhati.2011.lpd')
ts_list = D.get_timeseries(D.get_all_dataset_names())
df = D.get_timeseries_essentials()

D2 = LiPD()
D2.load_from_dir('Pages2k/')
df2 = D2.get_timeseries_essentials()
df_temp = df2[df2['paleoData_variableName'] == 'temperature']
df_filt = df2.query("paleoData_variableName in ('temperature','MXD','d18O')")

import matplotlib.pyplot as plt
df_filt.plot()
plt.show()
"""

EXEMPLAR_PYLIPD_ANSWER = '{"D": "D", "D2": "df_filt"}'


EXEMPLAR_PYLEOTUPS_CODE = """\
import pyleotups as pt

ds = pt.PangaeaDataset()
ds.search_studies(study_ids=830587)
bib, df_pub = ds.get_publications()

ds2 = pt.NOAADataset()
ds2.search_studies(noaa_id=33213)

ds3 = pt.NOAADataset()
ds3.search_studies(noaa_id=36778)

ds_combined = ds2 + ds3

import matplotlib.pyplot as plt
df_pub.plot()
plt.show()
"""

EXEMPLAR_PYLEOTUPS_ANSWER = '{"ds": "ds", "ds_combined": "ds_combined"}'


EXEMPLAR_LIPDGRAPH_CODE = """\
import requests
import pandas as pd
import numpy as np
import json
import io

url = 'https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic'
query = \"\"\"PREFIX le: <http://linked.earth/ontology#>
SELECT ?varID ?dataSetName ?lat ?lon ?val ?varunits WHERE {
    ?ds a le:Dataset . ?ds le:hasName ?dataSetName .
    ?ds le:hasPaleoData ?data .
    ?data le:hasMeasurementTable ?table .
    ?table le:hasVariable ?var .
    ?var le:hasValues ?val .
}\"\"\"

response = requests.post(url, data={'query': query})
data = io.StringIO(response.text)
df_res = pd.read_csv(data, sep=",")
df_res['val'] = df_res['val'].apply(lambda row: json.loads(row) if isinstance(row, str) else row)

df = df_res[~df_res['varID'].duplicated()]
filtered_df = df[df['timeval'].apply(lambda x: np.ptp(x) >= 1500)]
filtered_df2 = filtered_df[filtered_df['timeval'].apply(lambda x: abs(np.mean(np.diff(x))) <= 60)]

import pyleoclim as pyleo
ts_list = []
for _, row in filtered_df2.iterrows():
    ts_list.append(pyleo.GeoSeries(time=row['timeval'], value=row['val']))
mgs = pyleo.MultipleGeoSeries(ts_list)
pca = mgs.pca()
"""

EXEMPLAR_LIPDGRAPH_ANSWER = '{"df_res": "filtered_df2"}'

### Prompt Template with Exemplar

The same base prompt as `PROMPT_CODE_ONLY`, but with a worked example injected via `{exemplar_code}` and `{exemplar_answer}`. The exemplar is swapped out to test how similarity affects accuracy.

In [ ]:
PROMPT_WITH_EXEMPLAR = ChatPromptTemplate.from_template(
"""You are analyzing a Jupyter notebook to identify dataset variables used in a scientific analysis.

This notebook may load paleoclimate data from one or more of these sources:
- PyLiPD: LiPD() objects that load datasets via load(), load_remote_datasets(), or load_from_dir()
- PyleoTUPS: PangaeaDataset() or NOAADataset() objects that search for studies via search_studies()
- LiPDGraph: SPARQL queries sent via requests.post() to a LinkedEarth graph database endpoint, results parsed into DataFrames

For each dataset source in the notebook:
1. Identify the variable where data is first loaded or queried.
2. Trace the data flow and identify the final variable that holds the filtered or subsetted data actually used in the analysis.

Rules:
- If no filtering occurs, the final variable is the source variable itself.
- If datasets are combined (e.g. ds3 = ds1 + ds2), only return the combined variable, not the consumed inputs.
- If data flows through filtering (boolean indexing, subsetting, deduplication), return the last filtered variable before the data is consumed by analysis (PCA, plotting, modeling, etc.).
- Only return variables that hold dataset collections (DataFrames, dataset objects), not scalars, plots, or single rows.

Here is a worked example:

Notebook code:
```python
{exemplar_code}
```

Answer: {exemplar_answer}

Now analyze this notebook. Return ONLY a JSON object mapping each source variable to its final variable. No explanation.

Notebook code:
```python
{code}
```""")

### Exemplar Test Runner

In [ ]:
def run_exemplar_test(notebook_path: str, expected: dict):
    """Runs the exemplar prompt with all three exemplar types and compares results."""
    code = extract_code(notebook_path)
    chain = PROMPT_WITH_EXEMPLAR | llm

    exemplars = [
        ('PyLiPD (high similarity)',    EXEMPLAR_PYLIPD_CODE,    EXEMPLAR_PYLIPD_ANSWER),
        ('PyleoTUPS (medium similarity)', EXEMPLAR_PYLEOTUPS_CODE, EXEMPLAR_PYLEOTUPS_ANSWER),
        ('LiPDGraph (low similarity)',  EXEMPLAR_LIPDGRAPH_CODE, EXEMPLAR_LIPDGRAPH_ANSWER),
    ]

    print(f'=== {os.path.basename(notebook_path)} ===')
    print(f'Expected: {expected}')
    print()

    for label, ex_code, ex_answer in exemplars:
        resp = chain.invoke({
            'exemplar_code': ex_code,
            'exemplar_answer': ex_answer,
            'code': code,
        })
        print(f'  {label}:  {resp.content}')

    # Baseline: no exemplar (PROMPT_CODE_ONLY)
    baseline_chain = PROMPT_CODE_ONLY | llm
    baseline = baseline_chain.invoke({'code': code})
    print(f'  No exemplar (baseline):     {baseline.content}')
    print()

### Run Exemplar Tests

Each cell runs all three exemplar prompts (plus the no-exemplar baseline) on a single target notebook. Compare across rows to see how exemplar similarity affects accuracy.

| Target | Why |
|---|---|
| LIPD.ipynb | High-similarity exemplar matches the library; tests whether that helps |
| PyleoTUPS.ipynb | Medium-similarity exemplar matches; tests cross-library transfer |
| paleoPCA.ipynb | Low-similarity exemplar matches; tests whether filtering-pattern transfer works |

In [ ]:
run_exemplar_test('LIPD.ipynb', {
    'D_remote': 'D_remote',
    'D_url': 'D_url',
    'D': 'D',
    'D_dir': 'df_temp',
    'D_src': 'D_reloaded',
    'D_reloaded': 'D_reloaded',
})

In [ ]:
run_exemplar_test('PyleoTUPS.ipynb', {
    'ds': 'ds',
    'ds_sum': 'ds_sum',
    'noaa_ds': 'noaa_ds',
    'pangaea_ds': 'pangaea_ds',
})

In [ ]:
run_exemplar_test('paleoPCA.ipynb', {
    'df_res': 'filtered_df2',
})